# InferArena: calibrate the simulator against real vLLM

One-click version of [`docs/how-to-guides/calibrate-against-a-real-engine.md`](https://github.com/AndrewGumenyuk/InferArena/blob/main/docs/how-to-guides/calibrate-against-a-real-engine.md).

**Before you start:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Total time ~30 minutes (most of it model download + vLLM startup).

What this does: measures real per-token costs on GPU-backed vLLM, replays a seeded workload against it, runs the same workload through the calibrated InferArena simulator, and prints a side-by-side report.

In [ ]:
# 1. Verify the GPU is attached
!nvidia-smi

In [ ]:
# 2. Install vLLM and InferArena (~5 min)
!pip install -q vllm openai
!pip install -q "git+https://github.com/AndrewGumenyuk/InferArena.git"
!wget -q https://raw.githubusercontent.com/AndrewGumenyuk/InferArena/main/scripts/calibrate_against_vllm.py
print('install done')

In [ ]:
# 3. Start vLLM in the background (~10 min first run: model download)
!nohup vllm serve Qwen/Qwen2.5-0.5B --max-num-batched-tokens 2048 --max-num-seqs 64 > vllm.log 2>&1 &
print('server starting...')

In [ ]:
# 4. Wait until the server is ready
import time, urllib.request
for i in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2)
        print('vLLM is ready')
        break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError('vLLM did not start in 20 min — check vllm.log')

In [ ]:
# 5. Run the calibration (~10 min)
!python calibrate_against_vllm.py \
    --base-url http://localhost:8000/v1 \
    --model Qwen/Qwen2.5-0.5B \
    --max-tokens-per-step 2048 \
    --requests 32 --arrival-rate 2.0

In [ ]:
# 6. Show the report — share this output (e.g. paste it into a GitHub issue)
!cat inferarena_outputs/calibration/calibration_report.md